# Notebook 4:
### BI Aggregations, KPI Layer, and Dimensional Outputs

**Purpose**

Builds the Power BI semantic layer, including KPI summaries, date dimension, revenue aggregations, and cross table harmonization required for dashboard reporting.

#### BI outputs
- `date_table.csv`
- `kpi_summary.csv`
- `prescription_volume_by_day.csv`
- `prescription_volume_by_class_day.csv`
- `forecast_demand_by_class_day.csv`
- `revenue_by_insurance_type.csv`
- `inventory_turnover.csv`

#### Reference outputs
- `drug_class_map.csv`
- `ref_drugname_class_map.csv`

**Key responsibilitiy**
- Prepares clean, analysis ready fact tables and KPI summaries for the Power BI reporting layer.

In [1]:
# --- Setup: imports + paths ---
from pathlib import Path
import numpy as np
import pandas as pd

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_GEN = REPO_ROOT / "data" / "generated"
DATA_REAL = REPO_ROOT / "data" / "real"
FORECAST_DIR = REPO_ROOT / "forecast"
BI_DIR = REPO_ROOT / "bi_outputs"

for d in [DATA_GEN, FORECAST_DIR, BI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_GEN:", DATA_GEN)
print("FORECAST_DIR:", FORECAST_DIR)
print("BI_DIR:", BI_DIR)

REPO_ROOT: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization
DATA_GEN: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/data/generated
FORECAST_DIR: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/forecast
BI_DIR: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs


In [2]:
# --- Load required inputs ---
P_RX = DATA_GEN / "prescriptions.csv"
P_SALES = DATA_GEN / "sales.csv"
P_PRED = FORECAST_DIR / "predictions_daily_class.csv"

missing = [str(p) for p in [P_RX, P_SALES, P_PRED] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input file(s):\n" + "\n".join(missing))

rx = pd.read_csv(P_RX, low_memory=False)
sales = pd.read_csv(P_SALES, low_memory=False)
pred = pd.read_csv(P_PRED, low_memory=False)

print("prescriptions:", rx.shape)
print("sales:", sales.shape)
print("predictions_daily_class:", pred.shape)
print("pred columns:", list(pred.columns))

prescriptions: (12000, 9)
sales: (12000, 6)
predictions_daily_class: (10950, 3)
pred columns: ['DrugClass', 'Date', 'Demand']


In [3]:
# --- Build DrugID -> pharm_classes mapping (stable, weighted by forecast distribution) ---

pred_out = pred.rename(columns={"DrugClass": "pharm_classes", "Demand": "ForecastQty"}).copy()
pred_out["pharm_classes"] = pred_out["pharm_classes"].astype(str)

# Weight classes by total forecast, excluding Unmapped if present
class_weights = (
    pred_out.loc[pred_out["pharm_classes"].str.strip().ne("Unmapped")]
           .groupby("pharm_classes", as_index=True)["ForecastQty"]
           .sum()
           .astype(float)
           .clip(lower=0)
)

if class_weights.sum() == 0 or class_weights.empty:
    # Fallback: uniform
    classes = sorted(pred_out["pharm_classes"].dropna().unique().tolist())
    probs = np.ones(len(classes), dtype=float) / max(1, len(classes))
else:
    classes = class_weights.index.tolist()
    probs = (class_weights / class_weights.sum()).to_numpy()

def clean_id(s) -> str:
    s = str(s)
    s = s.replace("\u200b", "").replace("\ufeff", "").strip()
    s = "".join(ch for ch in s if ch.isdigit() or ch == "-")
    return s

rx_ids = rx["DrugID"].map(clean_id).dropna().astype(str)
unique_ids = pd.Series(sorted(rx_ids.unique()), name="DrugID_clean")

# Deterministic assignment: hash each DrugID_clean into [0,1) then map into cumulative weights
edges = np.cumsum(probs)
seed = np.uint64(42)

h = pd.util.hash_pandas_object(unique_ids, index=False).astype("uint64")
x = ((h ^ seed) % np.uint64(10**9)).astype("float64") / 1e9

idx = np.searchsorted(edges, x, side="right")
idx = np.clip(idx, 0, len(classes) - 1)

drug_class_map = pd.DataFrame({
    "DrugID_clean": unique_ids.values,
    "pharm_classes": np.asarray(classes, dtype=object)[idx]
})

P_MAP = DATA_GEN / "drug_class_map.csv"
drug_class_map.to_csv(P_MAP, index=False)

print("Wrote:", P_MAP, drug_class_map.shape)
print("Unmapped in map:", int((drug_class_map["pharm_classes"] == "Unmapped").sum()))
print(drug_class_map.head())

Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/data/generated/drug_class_map.csv (220, 2)
Unmapped in map: 0
   DrugID_clean                                      pharm_classes
0  0001-0001-11  Adrenergic beta2-Agonists, beta2-Adrenergic Ag...
1  0002-0002-22  Cytochrome P450 2D6 Inhibitors, Serotonin Reup...
2  0003-0003-33                Macrolide Antimicrobial, Macrolides
3  0004-0004-44  Angiotensin Converting Enzyme Inhibitor, Angio...
4  0005-0005-55  Angiotensin Converting Enzyme Inhibitor, Angio...


In [4]:
# --- Build actual prescriptions by day and by class/day ---
mapping = pd.read_csv(P_MAP, low_memory=False)

# Dates
date_col = "FillDate" if "FillDate" in rx.columns else ("Date" if "Date" in rx.columns else None)
if date_col is None:
    raise ValueError("prescriptions.csv needs FillDate or Date")

rx_work = rx.copy()
rx_work["DrugID_clean"] = rx_work["DrugID"].map(clean_id)
rx_work["Date"] = pd.to_datetime(rx_work[date_col], errors="coerce").dt.normalize()
rx_work = rx_work.loc[rx_work["Date"].notna()].copy()

rx_mapped = rx_work.merge(mapping, on="DrugID_clean", how="left")
rx_mapped["pharm_classes"] = rx_mapped["pharm_classes"].fillna("Unmapped")

print("Unmapped rows:", int(rx_mapped["pharm_classes"].eq("Unmapped").sum()), "of", len(rx_mapped))

# By class/day
if "RxID" in rx_mapped.columns:
    rx_by_class_day = (
        rx_mapped.groupby(["Date", "pharm_classes"], as_index=False)
                 .agg(Actual_Rx=("RxID", "nunique"))
    )
else:
    rx_by_class_day = (
        rx_mapped.groupby(["Date", "pharm_classes"], as_index=False)
                 .size()
                 .rename(columns={"size": "Actual_Rx"})
    )

# Total by day
if "RxID" in rx_mapped.columns:
    rx_by_day = (
        rx_mapped.groupby("Date", as_index=False)
                 .agg(TotalPrescriptions=("RxID", "nunique"))
    )
else:
    rx_by_day = (
        rx_mapped.groupby("Date", as_index=False)
                 .size()
                 .rename(columns={"size": "TotalPrescriptions"})
    )

P_OUT_CLASS = BI_DIR / "prescription_volume_by_class_day.csv"
P_OUT_DAY   = BI_DIR / "prescription_volume_by_day.csv"
rx_by_class_day.to_csv(P_OUT_CLASS, index=False)
rx_by_day.to_csv(P_OUT_DAY, index=False)

print("Wrote:", P_OUT_CLASS, rx_by_class_day.shape)
print("Wrote:", P_OUT_DAY, rx_by_day.shape)
rx_by_class_day.head()

Unmapped rows: 0 of 12000
Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/prescription_volume_by_class_day.csv (6985, 3)
Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/prescription_volume_by_day.csv (730, 2)


,Date,pharm_classes,Actual_Rx
0,2023-01-01,"Adrenergic beta2-Agonists, beta2-Adrenergic Ag...",2
1,2023-01-01,"Angiotensin Converting Enzyme Inhibitor, Angio...",2
2,2023-01-01,"Biguanide, Biguanides",2
3,2023-01-01,"Cytochrome P450 2D6 Inhibitors, Serotonin Reup...",1
4,2023-01-01,"Insulin Analog, Insulin",4


In [5]:
# --- Forecast by class/day scaled to match actual totals per day ---
pred_out = pred.rename(columns={"DrugClass": "pharm_classes", "Demand": "ForecastQty"}).copy()
pred_out["Date"] = pd.to_datetime(pred_out["Date"], errors="coerce").dt.normalize()
pred_out = pred_out.loc[pred_out["Date"].notna()].copy()

pred_out["ForecastQty"] = pd.to_numeric(pred_out["ForecastQty"], errors="coerce").fillna(0.0).clip(lower=0)

# Daily totals
pred_totals = pred_out.groupby("Date", as_index=False).agg(PredTotal=("ForecastQty","sum"))
actual_totals = rx_by_day.rename(columns={"TotalPrescriptions": "ActualTotal"}).copy()

scale = actual_totals.merge(pred_totals, on="Date", how="left")
scale["PredTotal"] = pd.to_numeric(scale["PredTotal"], errors="coerce").fillna(0.0)
scale["ScaleFactor"] = np.where(scale["PredTotal"] > 0, scale["ActualTotal"] / scale["PredTotal"], 1.0)

pred_scaled = pred_out.merge(scale[["Date","ScaleFactor"]], on="Date", how="left")
pred_scaled["ForecastQty"] = (pred_scaled["ForecastQty"] * pred_scaled["ScaleFactor"]).round().clip(lower=0).astype(int)

pred_final = pred_scaled[["Date","pharm_classes","ForecastQty"]].copy()

P_FCST = BI_DIR / "forecast_demand_by_class_day.csv"
pred_final.to_csv(P_FCST, index=False)

print("Wrote:", P_FCST, pred_final.shape)
pred_final.head()

Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/forecast_demand_by_class_day.csv (10950, 3)


,Date,pharm_classes,ForecastQty
0,2023-01-01,"Adrenergic beta2-Agonists, beta2-Adrenergic Ag...",1
1,2023-01-02,"Adrenergic beta2-Agonists, beta2-Adrenergic Ag...",1
2,2023-01-03,"Adrenergic beta2-Agonists, beta2-Adrenergic Ag...",0
3,2023-01-04,"Adrenergic beta2-Agonists, beta2-Adrenergic Ag...",3
4,2023-01-05,"Adrenergic beta2-Agonists, beta2-Adrenergic Ag...",1


In [6]:
# --- Revenue by insurance type ---
sales_work = sales.copy()

# Date
sales_work["Date"] = pd.to_datetime(sales_work["Date"], errors="coerce").dt.normalize()
sales_work = sales_work.loc[sales_work["Date"].notna()].copy()

sales_work["InsuranceType"] = sales_work.get("PaymentType", "Unknown").astype(str)
sales_work["Revenue"] = pd.to_numeric(sales_work.get("Revenue", 0), errors="coerce").fillna(0.0)

rev_by_ins_day = (
    sales_work.groupby(["Date","InsuranceType"], as_index=False)
              .agg(Revenue=("Revenue","sum"),
                   RxCount=("RxID","nunique"))
)

P_REV1 = BI_DIR / "revenue_by_insurance_type.csv"
P_REV2 = BI_DIR / "revenue_rx_by_insurance_day.csv"

rev_by_ins_day.to_csv(P_REV1, index=False)
rev_by_ins_day.to_csv(P_REV2, index=False)

print("Wrote:", P_REV1, rev_by_ins_day.shape)
print("Wrote:", P_REV2, rev_by_ins_day.shape)
rev_by_ins_day.head()

Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/revenue_by_insurance_type.csv (2741, 4)
Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/revenue_rx_by_insurance_day.csv (2741, 4)


,Date,InsuranceType,Revenue,RxCount
0,2023-01-01,Commercial,460.15,6
1,2023-01-01,Medicaid,435.33,3
2,2023-01-01,Medicare,1078.69,8
3,2023-01-02,Commercial,429.76,3
4,2023-01-02,Medicaid,428.92,4


In [7]:
# --- date_table.csv ---
def collect_date_range():
    candidates = []

    for p in BI_DIR.glob("*.csv"):
        candidates.append(p)

    candidates.append(P_RX)

    min_dt = None
    max_dt = None

    for p in candidates:
        try:
            df = pd.read_csv(p, nrows=2000)
        except Exception:
            continue

        date_cols = [c for c in df.columns if c.lower() in {"date","filldate","refilldate","timestamp","weekstartdate"}]
        if not date_cols:
            continue

        for col in date_cols:
            s = pd.to_datetime(df[col], errors="coerce")
            s = s[(s.dt.year >= 2000) & (s.dt.year <= 2100)]
            if s.notna().any():
                lo, hi = s.min(), s.max()
                min_dt = lo if (min_dt is None or lo < min_dt) else min_dt
                max_dt = hi if (max_dt is None or hi > max_dt) else max_dt

    if min_dt is None or max_dt is None:
        min_dt = pd.Timestamp("2023-01-01")
        max_dt = pd.Timestamp("2024-12-31")

    return min_dt.normalize(), max_dt.normalize()

min_dt, max_dt = collect_date_range()
dates = pd.date_range(min_dt, max_dt, freq="D")

date_table = pd.DataFrame({"Date": dates})
date_table["Year"] = date_table["Date"].dt.year
date_table["Month"] = date_table["Date"].dt.month
date_table["MonthName"] = date_table["Date"].dt.strftime("%b")
date_table["Quarter"] = date_table["Date"].dt.quarter
date_table["Week"] = date_table["Date"].dt.isocalendar().week.astype(int)
date_table["Day"] = date_table["Date"].dt.day
date_table["DayName"] = date_table["Date"].dt.strftime("%a")
date_table["IsWeekend"] = (date_table["Date"].dt.weekday >= 5).astype(int)

P_DATE = BI_DIR / "date_table.csv"
date_table.to_csv(P_DATE, index=False)
print("Wrote:", P_DATE, "range:", str(min_dt.date()), "to", str(max_dt.date()))
date_table.head()

Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/date_table.csv range: 2023-01-01 to 2024-12-31


,Date,Year,Month,MonthName,Quarter,Week,Day,DayName,IsWeekend
0,2023-01-01,2023,1,Jan,1,52,1,Sun,1
1,2023-01-02,2023,1,Jan,1,1,2,Mon,0
2,2023-01-03,2023,1,Jan,1,1,3,Tue,0
3,2023-01-04,2023,1,Jan,1,1,4,Wed,0
4,2023-01-05,2023,1,Jan,1,1,5,Thu,0


In [8]:
# --- KPI summary ---
def load_csv(path: Path):
    return pd.read_csv(path) if path.exists() else None

def pick_numeric_col(df, preferred):
    if df is None:
        return None
    cols = {c.lower(): c for c in df.columns}
    for n in preferred:
        if n.lower() in cols:
            return cols[n.lower()]
    nums = df.select_dtypes(include=[np.number]).columns.tolist()
    return nums[0] if nums else None

def safe_sum(df, col):
    if df is None or col is None or col not in df.columns:
        return None
    return float(pd.to_numeric(df[col], errors="coerce").fillna(0).sum())

def fmt(val, kind="number"):
    if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
        return "Needs input"
    if kind == "int":
        return int(round(val))
    if kind == "currency":
        return round(val, 2)
    return round(val, 4)

fcst_df = load_csv(BI_DIR / "forecast_demand_by_class_day.csv")
staff_df = load_csv(BI_DIR / "projected_staffing_by_hour.csv")
rev_df   = load_csv(BI_DIR / "revenue_by_insurance_type.csv")
turn_df  = load_csv(BI_DIR / "inventory_turnover.csv")

# Forecast
fcst_col = pick_numeric_col(fcst_df, ["ForecastQty", "Demand", "Qty"])
total_fcst = safe_sum(fcst_df, fcst_col)

# Staffing (use RequiredStaff as KPI)
staff_col = pick_numeric_col(staff_df, ["RequiredStaff", "ScheduledStaff", "OverUnder"])
total_staff = safe_sum(staff_df, staff_col)

# Revenue
rev_col = pick_numeric_col(rev_df, ["Revenue", "TotalRevenue"])
total_rev = safe_sum(rev_df, rev_col)

# Turnover
avg_turn = None
if turn_df is not None and len(turn_df):
    if "Turnover" in turn_df.columns:
        vals = pd.to_numeric(turn_df["Turnover"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        avg_turn = float(vals.mean()) if len(vals) else None

rows = [
    {"metric":"Total Forecast Demand", "value": fmt(total_fcst,"int"), "unit":"units", "source_file":"forecast_demand_by_class_day.csv"},
    {"metric":"Total Projected Staffing", "value": fmt(total_staff,"int"), "unit":"staff-units", "source_file":"projected_staffing_by_hour.csv"},
    {"metric":"Average Inventory Turnover", "value": fmt(avg_turn,"number"), "unit":"turns", "source_file":"inventory_turnover.csv"},
    {"metric":"Total Revenue", "value": fmt(total_rev,"currency"), "unit":"USD", "source_file":"revenue_by_insurance_type.csv"},
]

kpi_df = pd.DataFrame(rows)
P_KPI = BI_DIR / "kpi_summary.csv"
kpi_df.to_csv(P_KPI, index=False)

print("Wrote:", P_KPI)
kpi_df

Wrote: /Users/selenadavis/PythonProject1/PharmacyOperationsOptimization/bi_outputs/kpi_summary.csv


,metric,value,unit,source_file
0,Total Forecast Demand,1.192400e+04,units,forecast_demand_by_class_day.csv
1,Total Projected Staffing,1.336000e+04,staff-units,projected_staffing_by_hour.csv
2,Average Inventory Turnover,2.345000e-01,turns,inventory_turnover.csv
3,Total Revenue,1.520582e+06,USD,revenue_by_insurance_type.csv
